# Tone-timing tasks — listening notebook

Three experiments, all asking the same question in different clothes: **when several tones
repeat together, does displacing one of them get easier or harder as they drift out of step?**

Run the setup cell, then jump to whichever part you want.

| part | stimulus | axis | what varies |
|---|---|---|---|
| **1** | four tones, sheared | shear step, 0–100% | how far apart the onsets are staggered |
| **2** | two tones, each two partials | onset lag ΔT, 0–100% | synchrony → alternation |
| **3** | two pure tones | onset lag ΔT, 0–100% | the original, one sinusoid per tone |
| **4** | whatever you like | — | tone length, count, step, rate |
| **5** | results | — | what the three tasks measured |

**How the listening sections work.** Every section is one shift, with five jitters under it.
Each row is one trial: left column has the displaced tone, right column is the standard. They
are otherwise identical. Start at the bottom of a section (40 ms, obvious) and work up.

The plot in each section shows the stimulus at that shift, not the jitter — the jitter is a few
milliseconds and would be invisible.

Both clips in a row are scaled together, so what you hear is the real waveform pair. They can
still differ slightly in level: moving a tone out of a chord changes how much the tones sum.
That is in the experiment too, where a ±3 dB rove across trials stops it being usable as a cue.
Over one demo row you may hear it, so judge on timing.

Use the outline (☰, left) to navigate.

In [ ]:
#@title setup — run this first
import sys, subprocess, importlib.util, io, base64, wave, math
from pathlib import Path
REF='3fe4b728a7ef26404d1fcc92a5c872b26c687595'
REPO='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
try:
    import tshear, tcoh
except Exception:
    if bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab')):
        ROOT=Path('/content')/('seqsfg-'+REF[:12])
        if not ROOT.exists():
            subprocess.run(['git','clone','--no-checkout',REPO,str(ROOT)],check=True)
            subprocess.run(['git','-C',str(ROOT),'checkout','--detach',REF],check=True)
    else:
        ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'tshear/config.py').exists()),None)
    for _m in [k for k in list(sys.modules) if k.split('.')[0] in ('tshear','tcoh')]: del sys.modules[_m]
    sys.path.insert(0,str(ROOT))

get_ipython().run_line_magic('matplotlib','inline')
import numpy as np, matplotlib
matplotlib.rcParams.update({'figure.dpi':120,'font.size':8,
                            'axes.spines.top':False,'axes.spines.right':False})
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import HTML, display, Markdown

import tshear.config as TC, tshear.stimulus as TS
import tcoh.config as CC, tcoh.stimulus as CS

TARGET, OTHER, ACCENT = '#c0392b', '#5d6d7e', '#117864'
JITTERS = [2.0, 5.0, 10.0, 20.0, 40.0]

# ---- audio ----------------------------------------------------------------
def _wav(x, sr, peak):
    d = (np.clip(np.asarray(x, float) / max(peak, 1e-9) * 0.85, -1, 1) * 32767).astype('<i2')
    b = io.BytesIO()
    with wave.open(b, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(int(sr)); w.writeframes(d.tobytes())
    return base64.b64encode(b.getvalue()).decode()

def pairs(rows, sr, note=''):
    """rows: (label, displaced, standard). One trial per row, two columns.

    Both clips of a row are scaled by the SAME peak. Normalising each on its own would put up
    to 7 dB between the two columns -- moving a tone out of a chord changes how much the tones
    sum -- and you would pick the displaced one by loudness instead of by timing.
    """
    h = ['<table style="border-collapse:collapse;font:13px system-ui;width:100%;max-width:780px">',
         '<tr><th style="text-align:left;padding:4px 10px;width:80px">jitter</th>'
         '<th style="text-align:left;padding:4px 10px">displaced</th>'
         '<th style="text-align:left;padding:4px 10px">standard</th></tr>']
    for lab, a, b in rows:
        pk = max(float(np.abs(a).max()), float(np.abs(b).max()))
        h.append('<tr style="border-top:1px solid #e3e3e3">'
                 f'<td style="padding:6px 10px;color:#c0392b;font-weight:600">{lab}</td>'
                 f'<td style="padding:6px 10px"><audio controls preload="none" style="height:32px"'
                 f' src="data:audio/wav;base64,{_wav(a, sr, pk)}"></audio></td>'
                 f'<td style="padding:6px 10px"><audio controls preload="none" style="height:32px"'
                 f' src="data:audio/wav;base64,{_wav(b, sr, pk)}"></audio></td></tr>')
    h.append('</table>')
    if note: h.append(f'<div style="font:12px system-ui;color:#666;margin-top:6px">{note}</div>')
    display(HTML(''.join(h)))

# ---- tshear ---------------------------------------------------------------
SH = TC.Config()
SH_CONDS = {c.name: c for c in TC.conditions(SH)}

def shear_audio(step_pct, kind='figure'):
    cond = TC.Condition('x', step_pct, kind)
    out = []
    for j in JITTERS:
        tr = TS.build_trial(SH, cond, j, np.random.default_rng(int(j*7)),
                            target_position=1, direction=+1)
        out.append((f'{j:g} ms',
                    TS.render_interval(SH, tr.first, tr.phases),
                    TS.render_interval(SH, tr.second, tr.phases)))
    return out, SH.sample_rate

def shear_plot(step_pct, kind='figure', n_rep=3):
    fig, ax = plt.subplots(figsize=(7.4, 1.9))
    for rep in range(n_rep):
        o = TC.onsets_ms(SH, step_pct, rep) - SH.lead_ms
        for k in range(SH.n_tones):
            if kind == 'single' and k != SH.target_index: continue
            ax.add_patch(Rectangle((o[k], k-.34), SH.tone_ms, .68,
                                   color=TARGET if k == SH.target_index else OTHER,
                                   alpha=.95 if k == SH.target_index else .72))
    ax.set_xlim(-30, n_rep*SH.period_ms+SH.figure_span_ms(step_pct)+40)
    ax.set_ylim(-.7, SH.n_tones-.3)
    ax.set_yticks(range(SH.n_tones))
    ax.set_yticklabels([f'{f:.0f}' for f in SH.freqs_hz])
    ax.set_xlabel('ms'); ax.grid(axis='x', alpha=.15)
    plt.show()

# ---- tcoh -----------------------------------------------------------------
import tcoh
_BASE = Path(tcoh.__file__).resolve().parent.parent
def _cfg(rel):
    import json as _j
    return CC.Config.from_dict(_j.loads((_BASE/rel).read_text()))
CPX = _cfg('tcoh/configs/tcoh_complex.json').replace(interleaved_pcts=(0.0, 100.0))
PUR = _cfg('tcoh/configs/tcoh_curve_v1.json')

def coh_audio(cfg, lag_pct, interleaved=False):
    cond = CC.Condition('x', lag_pct, 'coherent', 'yoked', interleaved=interleaved)
    d = CC.validate(cfg); out = []
    for j in JITTERS:
        jj = min(j, cfg.delta_max_ms)
        tr = CS.build_trial(cfg, cond, jj, np.random.default_rng(int(jj*7)),
                            target_position=1, direction=-1)
        out.append((f'{jj:g} ms',
                    CS.render_interval(cfg, tr.first, d, tr.phases),
                    CS.render_interval(cfg, tr.second, d, tr.phases)))
    return out, cfg.sample_rate

def coh_plot(cfg, lag_pct, interleaved=False, n_rep=4):
    a_f, b_f = cfg.tone_freqs(interleaved)
    allf = sorted(a_f + b_f)
    lag = cfg.lag_ms(lag_pct)
    fig, ax = plt.subplots(figsize=(7.4, 1.9))
    for rep in range(n_rep):
        t = rep*cfg.soa_ms
        for f in b_f:
            ax.add_patch(Rectangle((t, allf.index(f)-.34), cfg.tone_ms, .68,
                                   color=TARGET, alpha=.95))
        for f in a_f:
            ax.add_patch(Rectangle((t+lag, allf.index(f)-.34), cfg.tone_ms, .68,
                                   color=OTHER, alpha=.72))
    ax.set_xlim(-30, n_rep*cfg.soa_ms+lag+40); ax.set_ylim(-.7, len(allf)-.3)
    ax.set_yticks(range(len(allf))); ax.set_yticklabels([f'{f:.0f}' for f in allf])
    ax.set_xlabel('ms'); ax.grid(axis='x', alpha=.15)
    plt.show()

def head(s): display(Markdown(s))
print(f"ready  ·  tshear {SH.n_tones} tones at {SH.rate_hz:g} Hz  ·  "
      f"tcoh complex {CPX.tone_ms:g}/{CPX.soa_ms:g} ms  ·  tcoh pure {PUR.tone_ms:g}/{PUR.soa_ms:g} ms")

---
# 1 · Four tones, sheared

Four tones repeat at 3 Hz. Their onsets are staggered by a fixed **step**: at 0% they are a
chord, at 100% an even arpeggio. The red tone (2241 Hz) is the one that gets displaced, always.

Measured thresholds for L01 are in part 5. The short version: 1.9 ms at step 0, about 18 ms
everywhere else.

## step 0%

All four start together. This is where the listener was 12× more sensitive.

In [ ]:
shear_plot(0.0)
rows, sr = shear_audio(0.0)
pairs(rows, sr)

## step 15%

12.5 ms of stagger. The advantage is already almost gone.

In [ ]:
shear_plot(15.0)
rows, sr = shear_audio(15.0)
pairs(rows, sr)

## step 30%

25 ms — the tones no longer overlap.

In [ ]:
shear_plot(30.0)
rows, sr = shear_audio(30.0)
pairs(rows, sr)

## step 50%

41.7 ms.

In [ ]:
shear_plot(50.0)
rows, sr = shear_audio(50.0)
pairs(rows, sr)

## step 75%

62.5 ms.

In [ ]:
shear_plot(75.0)
rows, sr = shear_audio(75.0)
pairs(rows, sr)

## step 100%

83.3 ms. The four onsets are now evenly spaced — a 12 Hz train.

In [ ]:
shear_plot(100.0)
rows, sr = shear_audio(100.0)
pairs(rows, sr)

## the target tone alone

No figure at all — just the 2241 Hz tone repeating at 3 Hz. This is the ceiling every condition
above is measured against: 22.7 ms for L01.

In [ ]:
shear_plot(0.0, kind='single')
rows, sr = shear_audio(0.0, kind='single')
pairs(rows, sr)

---
# 2 · Two tones, each made of two partials

A and B are complexes rather than sinusoids: 801 + 1489 Hz and 2754 + 4957 Hz, chosen so no pair
shares an auditory filter and no fundamental explains all four. 100 ms tones, 200 ms SOA.

**Separated** gives A the low pair and B the high pair. **Interleaved** alternates them, so no
frequency boundary separates A from B — same partials, swapped.

Red = B, the tone that moves. ΔT is its onset lag as a percentage of half the period.

## separated · ΔT 0%

A and B strictly simultaneous.

In [ ]:
coh_plot(CPX, 0.0)
rows, sr = coh_audio(CPX, 0.0)
pairs(rows, sr)

## separated · ΔT 50%

In [ ]:
coh_plot(CPX, 50.0)
rows, sr = coh_audio(CPX, 50.0)
pairs(rows, sr)

## separated · ΔT 75%

In [ ]:
coh_plot(CPX, 75.0)
rows, sr = coh_audio(CPX, 75.0)
pairs(rows, sr)

## separated · ΔT 87.5%

In [ ]:
coh_plot(CPX, 87.5)
rows, sr = coh_audio(CPX, 87.5)
pairs(rows, sr)

## separated · ΔT 100%

Exact alternation — the combined onset train is isochronous.

In [ ]:
coh_plot(CPX, 100.0)
rows, sr = coh_audio(CPX, 100.0)
pairs(rows, sr)

## interleaved · ΔT 0%

Same four partials as above, reassigned: A = 801 + 2754, B = 1489 + 4957.

In [ ]:
coh_plot(CPX, 0.0, interleaved=True)
rows, sr = coh_audio(CPX, 0.0, interleaved=True)
pairs(rows, sr)

## interleaved · ΔT 100%

Same four partials as above, reassigned: A = 801 + 2754, B = 1489 + 4957.

In [ ]:
coh_plot(CPX, 100.0, interleaved=True)
rows, sr = coh_audio(CPX, 100.0, interleaved=True)
pairs(rows, sr)

---
# 3 · Two pure tones

The original. One sinusoid per tone — 1000 Hz and 2378 Hz, 15 semitones apart — 75 ms tones at
150 ms SOA. Red is B, the tone that moves.

## ΔT 0%

In [ ]:
coh_plot(PUR, 0.0)
rows, sr = coh_audio(PUR, 0.0)
pairs(rows, sr)

## ΔT 25%

In [ ]:
coh_plot(PUR, 25.0)
rows, sr = coh_audio(PUR, 25.0)
pairs(rows, sr)

## ΔT 50%

In [ ]:
coh_plot(PUR, 50.0)
rows, sr = coh_audio(PUR, 50.0)
pairs(rows, sr)

## ΔT 75%

In [ ]:
coh_plot(PUR, 75.0)
rows, sr = coh_audio(PUR, 75.0)
pairs(rows, sr)

## ΔT 87.5%

In [ ]:
coh_plot(PUR, 87.5)
rows, sr = coh_audio(PUR, 87.5)
pairs(rows, sr)

## ΔT 100%

In [ ]:
coh_plot(PUR, 100.0)
rows, sr = coh_audio(PUR, 100.0)
pairs(rows, sr)

---
# 4 · Build your own

Set the numbers, run the cell. Tones may overlap — nothing here stops you making the rate faster
than the tones are long.

In [ ]:
#@title build a figure
tone_ms        = 60    #@param {type:"number"}
n_tones        = 4     #@param {type:"integer"}
step_ms        = 25    #@param {type:"number"}
rate_hz        = 3.0   #@param {type:"number"}
n_repeats      = 5     #@param {type:"integer"}
jitter_ms      = 20    #@param {type:"number"}
lowest_hz      = 683   #@param {type:"number"}
spacing_st     = 10.2  #@param {type:"number"}
target_tone    = 3     #@param {type:"integer"}

SR = 48000
period = 1000.0/rate_hz
freqs  = [lowest_hz*2**(spacing_st*k/12) for k in range(n_tones)]
k      = min(max(target_tone,1), n_tones)-1
span   = (n_tones-1)*step_ms
total  = 200 + (n_repeats-1)*period + span + tone_ms + abs(jitter_ms) + 200

def render(shift):
    n = int(round(total*SR/1000)); x = np.zeros(n)
    w = int(round(tone_ms*SR/1000)); r = max(int(round(10*SR/1000)), 1)
    env = np.ones(w); env[:r] = np.linspace(0,1,r); env[-r:] = np.linspace(1,0,r)
    t = np.arange(w)/SR
    for rep in range(n_repeats):
        for j, f in enumerate(freqs):
            on = 200 + rep*period + j*step_ms + (shift if (rep==n_repeats-1 and j==k) else 0.0)
            i = int(round(on*SR/1000)); e = min(n, i+w)
            if i < 0 or i >= n: continue
            x[i:e] += env[:e-i]*np.sin(2*np.pi*f*t[:e-i] + 2*np.pi*np.random.rand())
    return x

overlap = tone_ms > step_ms and step_ms > 0
print(f"{n_tones} tones {'+'.join(f'{f:.0f}' for f in freqs)} Hz")
print(f"period {period:.1f} ms · figure spans {span:.1f} ms · "
      f"{'tones OVERLAP in time' if overlap else 'tones do not overlap'}"
      + (f" · figure is LONGER than the period ({span+tone_ms:.0f} > {period:.0f} ms)"
         if span+tone_ms > period else ''))
fig, ax = plt.subplots(figsize=(7.4, 1.9))
for rep in range(min(n_repeats,4)):
    for j in range(n_tones):
        ax.add_patch(Rectangle((rep*period + j*step_ms, j-.34), tone_ms, .68,
                               color=TARGET if j==k else OTHER, alpha=.95 if j==k else .72))
ax.set_xlim(-30, min(n_repeats,4)*period+span+40); ax.set_ylim(-.7, n_tones-.3)
ax.set_yticks(range(n_tones)); ax.set_yticklabels([f'{f:.0f}' for f in freqs])
ax.set_xlabel('ms'); ax.grid(axis='x', alpha=.15); plt.show()
pairs([(f'{jitter_ms:g} ms', render(jitter_ms), render(0.0))], SR)

---
# 5 · Results

One listener each, one track per condition. A single track's 95% interval spans roughly a factor
of 3.7, so read the big differences and ignore the small ones.

In [ ]:
#@title the three curves
import numpy as np, matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

SHEAR = {0:1.89, 15:17.98, 30:16.02, 50:16.97, 75:20.18, 100:20.18}
SHEAR_SOLO = 22.65
CPLX  = {0:1.26, 50:24.03, 75:14.29, 87.5:12.01, 100:15.14}
CPLX_I = {0:0.80, 100:9.00}
PURE  = {0:3.54, 25:8.41, 50:14.98, 75:22.36, 100:17.24}

fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
def fmt(a, xl):
    a.set_yscale('log'); a.yaxis.set_minor_formatter(NullFormatter())
    a.set_yticks([1,2,5,10,20,30]); a.set_yticklabels(['1','2','5','10','20','30'])
    a.set_ylim(0.6, 40); a.grid(alpha=.25); a.set_xlabel(xl)

a = ax[0]
a.plot(list(SHEAR), list(SHEAR.values()), 'o-', color='#1a5276', lw=2, ms=7)
a.axhline(SHEAR_SOLO, color='#c0392b', ls='--', lw=1.6)
a.text(50, SHEAR_SOLO*1.12, 'tone alone', color='#c0392b', ha='center', fontsize=8)
a.set_xticks(list(SHEAR)); fmt(a, 'shear step (%)')
a.set_ylabel('jitter threshold (ms)'); a.set_title('four tones, sheared  (L01)', fontsize=9)

a = ax[1]
a.plot(list(CPLX), list(CPLX.values()), 'o-', color='#1a5276', lw=2, ms=7, label='separated')
a.plot(list(CPLX_I), list(CPLX_I.values()), 'D', color='#c0392b', ms=8, label='interleaved')
a.set_xticks([0,50,75,100]); a.invert_xaxis(); fmt(a, 'ΔT (%)')
a.set_title('two complex tones  (P01)', fontsize=9); a.legend(frameon=False, fontsize=7.5)

a = ax[2]
a.plot(list(PURE), list(PURE.values()), 'o-', color='#1a5276', lw=2, ms=7)
a.set_xticks(list(PURE)); a.invert_xaxis(); fmt(a, 'ΔT (%)')
a.set_title('two pure tones  (P01)', fontsize=9)
plt.tight_layout(); plt.show()

**Four tones, sheared.** 1.89 ms at step 0 against 22.65 ms for the same tone alone — a 12×
benefit from the other three. By 15% shear it is already 79% gone, and from there to 100% the
curve is flat (16.0–20.2 ms, a factor of 1.26 against a 3.65× noise band). The benefit is a
cliff at synchrony, not a gradient. The model predicts a gradient.

**Two complex tones.** 1.26 ms at synchrony, better than the pure-tone version, and interleaving
the partials made it easier rather than harder at both lags tested — 1.6× at ΔT 0 and 1.7× at
100%. Two single tracks, so treat the direction as a hint.

**Two pure tones.** A clean rise to 75%, then a drop at 100%. Bootstrapping the difference gives
1.28× with a 95% interval of [0.54, 2.39] — not established. ΔT 100% is the only lag whose
combined onset train is isochronous, which is the obvious suspect and why the newer designs
sample just below it.

Caveats that apply to all three: one listener, one track per condition, and the levels were
measured at 55 dB SPL rather than the 65 dB the configs target.